# U2T01 - Adapting BERT for NLP Tasks (Versión V2 Alta Precisión con Gráficas)
## Task: Part-of-Speech (POS) Tagging
**Autor:** Rivaldo  
**Hardware:** GPU Local (CUDA - NVIDIA GeForce RTX 4050 Laptop GPU)  
**Modelo Base:** `bert-base-cased` (110M parámetros)  
**Dataset:** Universal Dependencies (`universal-dependencies/universal_dependencies`, config `en_ewt`, 17 UPOS tags)

---

### 1. Entorno, Librerías y Verificación de CUDA
Configuramos las semillas aleatorias para garantizar reproducibilidad y verificamos que PyTorch reconozca la GPU para aceleración por CUDA.

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    set_seed
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Configurar estilo estético para las gráficas
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14

# Configurar semilla para reproducibilidad
SEED = 42
def setup_reproducibility(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

setup_reproducibility()

# Verificación estricta de CUDA GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 Dispositivo detectado: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU Modelo: {torch.cuda.get_device_name(0)}")
    print(f"⚡ Cantidad de GPUs: {torch.cuda.device_count()}")
    print(f"💾 VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ ADVERTENCIA: CUDA no está disponible. Revisa los controladores de tu GPU.")

KeyboardInterrupt: 

### 2. Carga del Dataset y Mapeo de Etiquetas UPOS
Cargamos el dataset Universal Dependencies (`en_ewt`) que contiene 17 etiquetas universales de Part-of-Speech.

In [ ]:
# Cargar dataset
dataset = load_dataset('universal-dependencies/universal_dependencies', 'en_ewt')
print("📊 Resumen del Dataset:", dataset)

# Extraer las 17 etiquetas UPOS únicas del split de entrenamiento
unique_tags = sorted(list(set(tag for sample in dataset['train'] for tag in sample['upos'])))
num_labels = len(unique_tags)

label2id = {tag: i for i, tag in enumerate(unique_tags)}
id2label = {i: tag for i, tag in enumerate(unique_tags)}

print(f"\n🏷️ Cantidad de etiquetas POS ({num_labels}): {unique_tags}")

### 3. Tokenizador y Alineación de Subpalabras (Regla del `-100`)
BERT utiliza WordPiece tokenization que divide palabras en subpalabras. Aplicamos la regla estricta requerida:
- La **primera subpalabra** recibe la etiqueta POS real.
- **Todas las subpalabras de continuación**, tokens especiales (`[CLS]`, `[SEP]`) y `padding` reciben la etiqueta `-100` (para ser ignoradas por `CrossEntropyLoss`).

In [ ]:
MODEL_CHECKPOINT = 'bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )
    
    labels = []
    for i, label in enumerate(examples['upos']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                # Tokens especiales ([CLS], [SEP], padding)
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Primera subpalabra de la palabra -> Etiqueta real
                label_ids.append(label2id[label[word_idx]])
            else:
                # Subpalabras de continuación -> -100
                label_ids.append(-100)
            previous_word_idx = word_idx
            
        labels.append(label_ids)
        
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

### 4. Celda de Sanity-Check (Obligatoria por Rúbrica)
Imprimimos un lote procesado mostrando cada token junto con su etiqueta alineada antes de comenzar el entrenamiento.

In [ ]:
# Probar con una muestra de entrenamiento
sample_raw = dataset['train'][:2]
sample_tokenized = tokenize_and_align_labels(sample_raw)

print("="*70)
print("🔍 SANITY-CHECK: ALINEACIÓN DE TOKENS Y ETIQUETAS (REGLA DEL -100)")
print("="*70)
tokens = tokenizer.convert_ids_to_tokens(sample_tokenized['input_ids'][0])
labels = sample_tokenized['labels'][0]

print(f"{'ID':<5} | {'Subword Token':<18} | {'Label ID':<10} | {'Tag / Status'}")
print("-"*70)
for idx, (tok, lab) in enumerate(zip(tokens, labels)):
    tag_str = id2label[lab] if lab != -100 else "[IGNORADO (-100)]"
    print(f"{idx:<5} | {tok:<18} | {lab:<10} | {tag_str}")
print("="*70)

### 5. Preprocesamiento Completo del Dataset

In [ ]:
print("🔄 Procesando todo el dataset...")
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset['train'].column_names
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("✅ Dataset procesado correctamente.")

### 6. Función de Métricas de Evaluación
Calculamos Accuracy, F1-score, Precision y Recall ignorando estrictamente las posiciones con etiqueta `-100` mediante `scikit-learn`.

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Extraer predicciones y etiquetas reales ignorando la etiqueta -100
    flat_preds = [
        pred_item for pred_seq, label_seq in zip(predictions, labels)
        for pred_item, label_item in zip(pred_seq, label_seq) if label_item != -100
    ]
    flat_labels = [
        label_item for label_seq in labels
        for label_item in label_seq if label_item != -100
    ]

    precision, recall, f1, _ = precision_recall_fscore_support(
        flat_labels, flat_preds, average='macro', zero_division=0
    )
    acc = accuracy_score(flat_labels, flat_preds)
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

--- 
### 7. MÉTODO 1: Partial Fine-Tuning (Head + Top 2 Encoder Layers Unfrozen)
En este método descongelamos la cabeza de clasificación (`classifier`) y las **últimas 2 capas del Encoder** (`layer.10` y `layer.11`, $\sim 14.8$M de parámetros) para permitir la adaptación sintáctica de alto nivel recomendada en la rúbrica del proyecto.

In [ ]:
print("🚀 Inicializando Método 1: Partial Fine-Tuning (Head + Top 2 Encoder Layers)...")
setup_reproducibility()

model_partial = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Descongelar las últimas 2 capas (10 y 11) + la cabeza de clasificación
for name, param in model_partial.named_parameters():
    if "classifier" in name or "bert.encoder.layer.10" in name or "bert.encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable_params_m1 = sum(p.numel() for p in model_partial.parameters() if p.requires_grad)
total_params_m1 = sum(p.numel() for p in model_partial.parameters())
print(f"🔓 Parámetros entrenables Método 1: {trainable_params_m1:,} de {total_params_m1:,} ({100 * trainable_params_m1 / total_params_m1:.2f}%)")

args_partial = TrainingArguments(
    output_dir="./results_partial",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,  # LR optimizado para capas superiores
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=SEED,
    logging_steps=50,
    report_to="none"
)

trainer_partial = Trainer(
    model=model_partial,
    args=args_partial,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

start_time = time.time()
trainer_partial.train()
m1_train_time = time.time() - start_time
print(f"⏱️ Tiempo de entrenamiento Método 1: {m1_train_time:.2f} segundos")

# Evaluación final en Test Set
m1_eval = trainer_partial.evaluate(tokenized_datasets["test"])
print("📊 Resultados en Test Set (Método 1 - Partial Fine-tuning):", m1_eval)

--- 
### 8. MÉTODO 2: Full Fine-Tuning con 5 Épocas, Warmup y Scheduler Cosine (Alta Precisión)
En este método entrenamos todo el modelo (BERT + Head) a lo largo de 5 épocas agregando `warmup_ratio=0.1` y desintegración suave `cosine` para maximizar la precisión final.

In [ ]:
print("🚀 Inicializando Método 2: Full Fine-Tuning (5 Épocas + Cosine Scheduler)...")
setup_reproducibility()

model_full = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

trainable_params_m2 = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
print(f"🔓 Parámetros entrenables Método 2: {trainable_params_m2:,} de {total_params_m1:,} (100%)")

# Configurar Dos Grupos de Parámetros (Parameter Groups)
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model_full.named_parameters() if "classifier" in n],
        "lr": 1e-3,  # LR alto para la cabeza recién inicializada
    },
    {
        "params": [p for n, p in model_full.named_parameters() if "classifier" not in n],
        "lr": 2e-5,  # LR bajo para el encoder preentrenado
    },
]

optimizer_full = torch.optim.AdamW(optimizer_grouped_parameters, weight_decay=0.01)

args_full = TrainingArguments(
    output_dir="./results_full",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,           # 5 Épocas para precisión máxima
    warmup_steps=100,             # Warmup 10%
    lr_scheduler_type="cosine",   # Planificador Cosine
    seed=SEED,
    logging_steps=50,
    report_to="none"
)

trainer_full = Trainer(
    model=model_full,
    args=args_full,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer_full, None)
)

start_time = time.time()
trainer_full.train()
m2_train_time = time.time() - start_time
print(f"⏱️ Tiempo de entrenamiento Método 2: {m2_train_time:.2f} segundos")

# Evaluación final en Test Set
m2_eval = trainer_full.evaluate(tokenized_datasets["test"])
print("📊 Resultados en Test Set (Método 2 - Full Fine-tuning):", m2_eval)

--- 
### 9. Tabla Comparativa y Gráficas Visuales de Rendimiento
Consolidamos la comparación entre ambos métodos tanto de forma tabular como visual (curvas de pérdida, métricas comparativas y matriz de confusión).

In [ ]:
comparison_data = {
    "Método de Adaptación": ["Partial Fine-Tuning (Top 2 Layers)", "Full Fine-Tuning (5 Épocas + Cosine)"],
    "Parámetros Entrenables": [f"{trainable_params_m1:,}", f"{trainable_params_m2:,}"],
    "Tiempo Entren. (seg)": [round(m1_train_time, 2), round(m2_train_time, 2)],
    "Test Loss": [round(m1_eval["eval_loss"], 4), round(m2_eval["eval_loss"], 4)],
    "Test Accuracy": [f"{m1_eval.get('eval_accuracy', 0)*100:.2f}%", f"{m2_eval.get('eval_accuracy', 0)*100:.2f}%"],
    "Test F1-Score": [round(m1_eval["eval_f1"], 4), round(m2_eval["eval_f1"], 4)]
}

df_comparison = pd.DataFrame(comparison_data)
print("="*80)
print("🏆 TABLA COMPARATIVA EMPÍRICA - RIVALDO (POS TAGGING)")
print("="*80)
print(df_comparison.to_string(index=False))
print("="*80)

#### 9.1. Gráfica 1: Curvas de Pérdida de Entrenamiento (Training Loss vs Steps)

In [ ]:
# Extraer logs de entrenamiento
m1_log = [log for log in trainer_partial.state.log_history if "loss" in log]
m2_log = [log for log in trainer_full.state.log_history if "loss" in log]

m1_steps = [log["step"] for log in m1_log]
m1_losses = [log["loss"] for log in m1_log]

m2_steps = [log["step"] for log in m2_log]
m2_losses = [log["loss"] for log in m2_log]

plt.figure(figsize=(10, 5))
plt.plot(m1_steps, m1_losses, label="Método 1: Partial Fine-tuning (Top 2 Layers)", color='#e74c3c', linewidth=2.5, marker='o')
plt.plot(m2_steps, m2_losses, label="Método 2: Full Fine-tuning (5 Épocas + Cosine)", color='#2ecc71', linewidth=2.5, marker='s')
plt.title("📉 Comparativa de Pérdida durante el Entrenamiento (Training Loss)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Pasos de Entrenamiento (Steps)")
plt.ylabel("CrossEntropy Loss")
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

#### 9.2. Gráfica 2: Comparativa Visual de Métricas de Evaluación (Accuracy, F1, Precision, Recall)

In [ ]:
metrics_names = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
m1_scores = [m1_eval['eval_accuracy'], m1_eval['eval_f1'], m1_eval['eval_precision'], m1_eval['eval_recall']]
m2_scores = [m2_eval['eval_accuracy'], m2_eval['eval_f1'], m2_eval['eval_precision'], m2_eval['eval_recall']]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, [s*100 for s in m1_scores], width, label='Método 1 (Partial Top 2)', color='#3498db')
rects2 = ax.bar(x + width/2, [s*100 for s in m2_scores], width, label='Método 2 (Full 5 Épocas)', color='#2ecc71')

ax.set_ylabel('Porcentaje / Score (%)', fontweight='bold')
ax.set_title('📊 Comparación de Métricas en el Test Set', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontweight='bold')
ax.set_ylim(70, 105)
ax.legend(frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.6)

for rect in rects1:
    height = rect.get_height()
    ax.annotate(f'{height:.2f}%',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=9, fontweight='bold')

for rect in rects2:
    height = rect.get_height()
    ax.annotate(f'{height:.2f}%',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

#### 9.3. Gráfica 3: Matriz de Confusión Normalizada del Modelo Ganador

In [ ]:
print("🔍 Generando Matriz de Confusión del Modelo Ganador...")
best_trainer = trainer_full if m2_eval["eval_f1"] >= m1_eval["eval_f1"] else trainer_partial
test_preds = best_trainer.predict(tokenized_datasets["test"])

preds_argmax = np.argmax(test_preds.predictions, axis=2)
labels_test = test_preds.label_ids

flat_preds = [
    pred_item for pred_seq, label_seq in zip(preds_argmax, labels_test)
    for pred_item, label_item in zip(pred_seq, label_seq) if label_item != -100
]
flat_labels = [
    label_item for label_seq in labels_test
    for label_item in label_seq if label_item != -100
]

cm = confusion_matrix(flat_labels, flat_preds, normalize='true')

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=unique_tags,
    yticklabels=unique_tags,
    cbar_kws={'label': 'Proporción de Acierto'}
)
plt.title('🧠 Matriz de Confusión Normalizada (17 Categorías UPOS)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Etiqueta Predicha por BERT', fontweight='bold')
plt.ylabel('Etiqueta Real (Ground Truth)', fontweight='bold')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

--- 
### 10. Exportación del Mejor Modelo y Tokenizador (Para Hugging Face Hub)
Guardamos localmente el modelo final seleccionado para que Damian o tú lo suban al Hugging Face Hub.

In [ ]:
OUTPUT_DIR = "./best_pos_bert_model"

# Elegir el modelo con mejor F1
if m2_eval["eval_f1"] >= m1_eval["eval_f1"]:
    print("🏆 Guardando Modelo Ganador: Método 2 (Full Fine-tuning)")
    trainer_full.save_model(OUTPUT_DIR)
else:
    print("🏆 Guardando Modelo Ganador: Método 1 (Partial Fine-tuning)")
    trainer_partial.save_model(OUTPUT_DIR)

tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Modelo y tokenizador guardados exitosamente en: {OUTPUT_DIR}")